# 03 — Frozen embeddings + image-quality features (KU-Optofil), Google Colab

**Thesis:** Certified Adaptive Examination of Peripheral Blood Smears. **Phase 3** (supports RQ1: Kill-tests 1–2, RQ3, RQ4)

**What this notebook does (and nothing else):**
1. Embeds every cell of `cells.csv` (28,280 cells, 275 patients) with **one frozen, publicly available encoder**. That encoder is not trained on KU-Optofil, so the embeddings cannot leak patient information.
2. Computes simple **image-quality features** per cell (brightness, contrast, blur, stain colour statistics) for the RQ4 analysis "does classifier error correlate with image quality?".
3. Runs a **sanity check**: a cross-fitted (patient-grouped, 5-fold) linear probe for blast vs. rest. This is *not* the thesis classifier; it only confirms the pipeline works.
4. Saves everything to Google Drive, together with a `config.json` that records versions, the model, preprocessing and checksums.

**Before running:** put these two files in Google Drive under `MyDrive/certified-adaptive-pbs/`:
- `dataset.zip` (from Zenodo 10.5281/zenodo.17333317)
- `cells.csv` (from `data/ku_optofil/cells.csv` in the repo, produced by `scripts/02_make_folds.py`)

**Runtime:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.
Expected time: about 5–10 min for the unzip plus about 5 min for the embeddings on a T4.

**Encoder choice (logged as a deviation):** InfoDPP-PAC uses Phikon, a histopathology (H&E) encoder. There is no equivalent open encoder established for Wright–Giemsa blood smears, so we use **DINOv2 ViT-S/14** (general-purpose self-supervised; `timm: vit_small_patch14_dinov2.lvd142m`) as the single selector embedding. The thesis does **not** compare encoders (out of scope).


In [ ]:
# 0. Setup: environment + GPU check
!pip -q install timm==1.0.* 2>/dev/null
import os, json, time, hashlib, platform, subprocess
import numpy as np, pandas as pd, torch, timm
print("torch", torch.__version__, "| timm", timm.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))
else: print("WARNING: no GPU -> this still works, but slowly (~30-60 min). Runtime > Change runtime type > T4 GPU")

In [ ]:
# 1. Paths (edit DRIVE_DIR only if you used a different folder)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/certified-adaptive-pbs'
OUT_DIR   = f'{DRIVE_DIR}/embeddings_v1'
LOCAL     = '/content/ku'
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(LOCAL, exist_ok=True)
for f in ['dataset.zip', 'cells.csv']:
    assert os.path.exists(f'{DRIVE_DIR}/{f}'), f'missing {DRIVE_DIR}/{f}'
print('inputs found')

In [ ]:
# 2. Copy the zip to local disk (much faster than reading from Drive), record its checksum, unzip
def sha256(p, bs=1<<22):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(bs), b''): h.update(b)
    return h.hexdigest()
t0 = time.time()
if not os.path.exists(f'{LOCAL}/dataset'):
    !cp "{DRIVE_DIR}/dataset.zip" /content/dataset.zip
    !unzip -q /content/dataset.zip -d {LOCAL}
ZIP_SHA256 = sha256('/content/dataset.zip')
print('sha256(dataset.zip) =', ZIP_SHA256, f'| {time.time()-t0:.0f}s')
# Compare this with data/ku_optofil/CHECKSUMS.txt on your laptop.

In [ ]:
# 3. Load cells.csv and resolve image paths
cells = pd.read_csv(f'{DRIVE_DIR}/cells.csv', dtype={'patient_id': str})
cells['abs_path'] = LOCAL + '/' + cells['rel_path']
missing = (~cells.abs_path.map(os.path.exists)).sum()
print(len(cells), 'cells |', cells.patient_id.nunique(), 'patients | blasts:', cells.is_blast.sum(), '| missing files:', missing)
assert missing == 0, 'paths do not match: check that the zip extracted to /content/ku/dataset/...'
print(cells.groupby('fold').agg(cells=('is_blast','size'), blasts=('is_blast','sum'), patients=('patient_id','nunique')))

In [ ]:
# 4. Frozen encoder + deterministic preprocessing
SEED = 20260919
torch.manual_seed(SEED); np.random.seed(SEED)
torch.backends.cudnn.benchmark = False; torch.backends.cudnn.deterministic = True
MODEL_NAME = 'vit_small_patch14_dinov2.lvd142m'
IMG_SIZE = 224   # 368x368 crops -> resized to 224 (16x16 patches of 14 px)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=0, img_size=IMG_SIZE).eval().to(device)
cfg_data = timm.data.resolve_data_config({}, model=model)
MEAN, STD = cfg_data['mean'], cfg_data['std']
print('embedding dim:', model.num_features, '| mean/std:', MEAN, STD)

In [ ]:
# 5. Dataset: returns the normalised tensor + image-quality features computed on the ORIGINAL crop
from PIL import Image
import torchvision.transforms.functional as TF
from scipy import ndimage

def quality_features(rgb):   # rgb: HxWx3 uint8
    x = rgb.astype(np.float32) / 255.0
    gray = x @ np.array([0.299, 0.587, 0.114], np.float32)
    lap = ndimage.laplace(gray)
    hsv = np.array(Image.fromarray(rgb).convert('HSV'), np.float32) / 255.0
    return dict(q_brightness=float(gray.mean()), q_contrast=float(gray.std()),
                q_blur_lapvar=float(lap.var()),                        # low = blurrier
                q_r=float(x[..., 0].mean()), q_g=float(x[..., 1].mean()), q_b=float(x[..., 2].mean()),
                q_sat=float(hsv[..., 1].mean()), q_hue=float(hsv[..., 0].mean()))

class Cells(torch.utils.data.Dataset):
    def __init__(self, paths): self.paths = list(paths)
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        im = Image.open(self.paths[i]).convert('RGB')
        q = quality_features(np.asarray(im))
        t = TF.to_tensor(TF.resize(im, [IMG_SIZE, IMG_SIZE], antialias=True))
        t = TF.normalize(t, MEAN, STD)
        return t, i, torch.tensor(list(q.values()), dtype=torch.float32)

Q_NAMES = list(quality_features(np.zeros((8, 8, 3), np.uint8)).keys())
dl = torch.utils.data.DataLoader(Cells(cells.abs_path), batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
# 6. Extract embeddings (row order == cells.csv row order; asserted below)
E = np.zeros((len(cells), model.num_features), np.float32); Qf = np.zeros((len(cells), len(Q_NAMES)), np.float32)
seen = np.zeros(len(cells), bool); t0 = time.time()
with torch.inference_mode():
    for b, (x, idx, q) in enumerate(dl):
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(device == 'cuda')):
            z = model(x.to(device, non_blocking=True))
        E[idx.numpy()] = z.float().cpu().numpy(); Qf[idx.numpy()] = q.numpy(); seen[idx.numpy()] = True
        if b % 50 == 0: print(f'batch {b}/{len(dl)}  {time.time()-t0:.0f}s')
assert seen.all() and np.isfinite(E).all()
print('done', E.shape, f'{time.time()-t0:.0f}s')

In [ ]:
# 7. Sanity check (NOT the thesis classifier): cross-fitted linear probe, blast vs rest
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
y = cells.is_blast.values; p_oof = np.zeros(len(cells))
for f in sorted(cells.fold.unique()):
    tr, te = cells.fold.values != f, cells.fold.values == f
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.1, class_weight='balanced'))
    clf.fit(E[tr], y[tr]); p_oof[te] = clf.predict_proba(E[te])[:, 1]
    print(f'fold {f}: test blasts={y[te].sum():4d}  AUC={roc_auc_score(y[te], p_oof[te]):.3f}')
print(f'pooled out-of-fold AUC = {roc_auc_score(y, p_oof):.4f}')
# per-source-patient blast recall at the threshold giving 95% specificity on non-blasts (out-of-fold)
thr = np.quantile(p_oof[y == 0], 0.95)
rec = cells.assign(p=p_oof)[y == 1].groupby('patient_id').p.apply(lambda s: (s >= thr).mean())
print('per-patient blast recall @95% spec:'); print(rec.round(3).to_string())

In [ ]:
# 8. Save outputs to Drive (+ config for reproducibility)
np.save(f'{OUT_DIR}/embeddings_dinov2_vits14.npy', E.astype(np.float16))
out = cells[['image_name', 'patient_id', 'cell_type', 'is_blast', 'fold', 'provided_split']].copy()
out[Q_NAMES] = Qf; out['probe_p_oof'] = p_oof
out.to_csv(f'{OUT_DIR}/cells_features.csv', index=False)
config = dict(notebook='03_embeddings_colab.ipynb', created_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
              model=MODEL_NAME, img_size=IMG_SIZE, mean=list(MEAN), std=list(STD), dtype_saved='float16',
              n_cells=int(len(cells)), dim=int(E.shape[1]), seed=SEED, zip_sha256=ZIP_SHA256,
              torch=torch.__version__, timm=timm.__version__, python=platform.python_version(),
              gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
              quality_features=Q_NAMES, probe=dict(type='logreg C=0.1 balanced, standardised', pooled_auc=float(roc_auc_score(y, p_oof))),
              note='Frozen encoder not trained on KU-Optofil; row order of the .npy == cells_features.csv == cells.csv')
json.dump(config, open(f'{OUT_DIR}/config.json', 'w'), indent=1)
print(json.dumps(config, indent=1)); print('\nSaved to', OUT_DIR)

## After it finishes
1. In Google Drive, download the folder `certified-adaptive-pbs/embeddings_v1/` (3 files).
2. Put the files in your repo at `results/embeddings_v1/` on your laptop.
3. The next steps (blast separability, the InfoDPP-PAC reimplementation, the kill-tests) run locally on the CPU.

Git tracks only `config.json` (the `.npy` and `.csv` are ignored), so every result can be traced back to exactly this run.
